# RDP Shortpath Setup – Analyse

Dieses Notebook analysiert den **dedizierten RDP-Shortpath-Session-Host** (`infra/shortpath/`) und prüft, ob der Public-Networks-Pfad (STUN/TURN) korrekt vorbereitet ist.

**Design-Entscheidung:** Public Networks (STUN/TURN) - UDP/TCP bleiben im Windows-Default. Der Managed-Networks-Listener auf UDP 3390 bleibt aus (siehe `infra/shortpath/MANAGED-NETWORKS.md`). Direct Stream (DS) ist der neue Name des bestehenden Streaming-Stacks, kein separater Deployment-Schalter.

**Client:** Native Windows App verwenden, nicht den Browser. Fuer DS Multipath mit UDP und redundanten TCP-Pfaden Windows App fuer Windows ab 2.0.1069.0 verwenden. Im Verbindungsdialog UDP / UDP (Relay) und Multipath pruefen.

**Was geprüft wird:**
1. Variablen + Login-Kontext
2. Existenz der Shortpath-Ressourcen (Host Pool, App Group, Workspace, VM, NIC)
3. Session-Host-Registrierung im Host Pool
4. RBAC der AVD-Gruppe auf den Shortpath-Ressourcen
5. Firewall-Relay-Regel (UDP 3478 → 51.5.0.0/16)
6. Effektive Routen + ausgehende IP der Shortpath-NIC
7. In-Guest: RDP-Transport-Default (UDP) + Managed-Listener-Status
8. Post-Connection-Verifikation (Event ID 135) – optional

> Kernel: **Bash** (Zellen-Sprache `shellscript`). Ausgaben werden nach `output/shortpath-*.json` gespeichert.


## 1. Variablen + Login-Kontext


In [ ]:
export PREFIX=cptdazavdvwan
export RG=rg-${PREFIX}
export SUB=${AZURE_SUBSCRIPTION_ID:-a0844269-41ae-442c-8277-415f1283d422}
az account set --subscription "$SUB"
export GROUP_ID=${AVD_USER_GROUP_OBJECT_ID:-23b3a9fd-22f3-461c-9dc8-7f3183e87d02}
export TEST_START_UTC=$(date -u +%Y-%m-%dT%H:%M:%SZ)

# Dedizierter Shortpath-Stack (infra/shortpath/)
export HP=hp-shortpath-${PREFIX}
export DAG=dag-shortpath-${PREFIX}
export WS=ws-shortpath-${PREFIX}
export VM=vm-shortpath-${PREFIX}
export NIC=nic-shortpath-${PREFIX}

# Standard-Stack zum Vergleich (infra/main.bicep)
export HP_BASE=hp-${PREFIX}
export VM_BASE=vm-avd-${PREFIX}
export NIC_BASE=nic-avd-${PREFIX}

mkdir -p output
echo "RG=$RG"
echo "SUB=$SUB"
echo "Shortpath : HP=$HP  VM=$VM  NIC=$NIC"
echo "Standard  : HP=$HP_BASE  VM=$VM_BASE  NIC=$NIC_BASE"


## 2. Shortpath-Ressourcen prüfen

**Erwartung:** Host Pool, App Group, Workspace, VM und NIC existieren und sind `Succeeded`.


In [ ]:
echo "=== Deployment 'shortpath' outputs ==="
az deployment group show -g $RG -n shortpath --query properties.outputs -o json | tee output/shortpath-outputs.json

echo ""
echo "=== Ressourcen-Inventar ==="
{
  az desktopvirtualization hostpool show -g $RG -n $HP --query "{type:type,name:name,state:provisioningState,poolType:hostPoolType,lb:loadBalancerType}" -o json
  az desktopvirtualization applicationgroup show -g $RG -n $DAG --query "{name:name,type:applicationGroupType}" -o json
  az desktopvirtualization workspace show -g $RG -n $WS --query "{name:name,appGroups:applicationGroupReferences}" -o json
  az vm show -g $RG -n $VM --query "{name:name,size:hardwareProfile.vmSize,computer:osProfile.computerName,state:provisioningState}" -o json
  az network nic show -g $RG -n $NIC --query "{name:name,privateIp:ipConfigurations[0].privateIPAddress,subnet:ipConfigurations[0].subnet.id}" -o json
} | tee output/shortpath-inventory.json


## 3. Session-Host-Registrierung

**Erwartung:** `vmsp01` ist im Host Pool registriert, `Status = Available`, `allowNewSession = true`.


In [ ]:
echo "=== Session Hosts in $HP ==="
# Diese az-CLI-Version hat kein 'sessionhost'-Kommando -> Management-API via az rest.
az rest --method GET \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.DesktopVirtualization/hostPools/${HP}/sessionHosts?api-version=2023-09-05" \
  --query "value[].{name:name,status:properties.status,allowNewSession:properties.allowNewSession,agentVersion:properties.agentVersion,osVersion:properties.osVersion}" -o json \
  | tee output/shortpath-sessionhosts.json


## 4. RBAC der AVD-Gruppe auf Shortpath-Ressourcen

**Erwartung:** `grp-avd-users-<prefix>` hat
- **Desktop Virtualization User** auf `dag-shortpath-`
- **Virtual Machine User Login** auf `vm-shortpath-`
- **Reader** auf der Resource Group (aus `main.bicep`)


In [ ]:
: "${GROUP_ID:?Zuerst die Variablen-Zelle ausfuehren}"
echo "AVD-Gruppe grp-avd-users-${PREFIX}: ${GROUP_ID:-<nicht gefunden>}"
echo ""
echo "=== Rollen der Gruppe im RG-Scope (gefiltert auf shortpath) ==="
az role assignment list --assignee "$GROUP_ID" --all \
  --query "[?contains(scope,'shortpath') || ends_with(scope,'rg-${PREFIX}')].{role:roleDefinitionName,scope:scope}" -o json \
  | tee output/shortpath-rbac.json


## 5. Firewall-Relay-Regel (Public-Networks-Pfad)

RDP Shortpath für Public Networks benötigt ausgehend **UDP 3478 → 51.5.0.0/16** (TURN-Relay).

**Erwartung:** Die Netzwerkregel `rdp-shortpath-relay` erlaubt UDP 3478 aus dem Spoke. Sie wird nur ausgewertet, wenn der Traffic die Firewall durchlaeuft; der aktuelle Service-Tag-Bypass schickt `51.5.0.0/16` ueber NAT Gateway.


In [ ]:
echo "=== Firewall-Netzwerkregel 'rdp-shortpath-relay' ==="
az network firewall policy rule-collection-group collection list \
  --policy-name afwp-${PREFIX} -g $RG --rule-collection-group-name rcg-allow \
  --query "[].rules[?name=='rdp-shortpath-relay'].{name:name,proto:ipProtocols,src:sourceAddresses,dst:destinationAddresses,ports:destinationPorts}[]" -o json \
  | tee output/shortpath-fw-rule.json

echo ""
echo "Hinweis: Quelle 10.1.0.0/16 (avdSpokeAddr) deckt den gesamten Spoke ab,"
echo "also auch nic-shortpath-${PREFIX} im Subnet snet-avd-hosts."


## 6. Effektive Routen + ausgehende IP

**Erwartung:** Der Shortpath-Host folgt demselben Routing wie der primaere Host. `51.5.0.0/16` nutzt aktuell Internet als Next Hop und die NAT-Gateway-PIP. Anderer Internet-Traffic folgt dem Default-Pfad durch die Firewall. Alle Praefixe der effektiven Routen pruefen.


In [ ]:
echo "=== Effektive Routen auf $NIC ==="
az network nic show-effective-route-table -g $RG -n $NIC -o json \
  | tee output/shortpath-effective-routes.json \
  | jq -r '.value[] | select(.source != null) | . as $route | .addressPrefix[] | "\(.)\t-> \($route.nextHopType)\t(\($route.source))"'


## 7. In-Guest: RDP-Transport-Default prüfen

Per `az vm run-command` wird auf dem Host der RDP-Transport-Status aus der Registry gelesen.

**Erwartung (Public Networks):**
- `UDP transport: DEFAULT (enabled)` → STUN/TURN nutzbar
- `Managed-networks listener: not set` → Managed-Pfad bewusst aus


In [ ]:
echo "=== RDP-Transport-Default auf $VM (dauert ~30s) ==="
az vm run-command invoke -g $RG -n $VM --command-id RunPowerShellScript --scripts "
  \$p='HKLM:\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\Terminal Services'
  \$ErrorActionPreference='Stop'
  \$v=if (Test-Path \$p) { Get-ItemProperty -Path \$p } else { \$null }
  if (\$null -eq \$v -or \$null -eq \$v.SelectTransport) {
    Write-Output 'UDP transport: DEFAULT (enabled). Public-networks Shortpath (STUN/TURN) can be used.'
  } elseif (\$v.SelectTransport -eq 1) {
    Write-Output 'UDP transport: DISABLED by session-host policy SelectTransport=1. Falls back to TCP.'
  } else {
    Write-Output ('UDP transport policy SelectTransport=' + \$v.SelectTransport)
  }
  Write-Output ('Managed-networks listener (fUseUdpPortRedirector): ' + \$(if (\$v.fUseUdpPortRedirector) { \$v.fUseUdpPortRedirector } else { 'not set (managed-networks Shortpath off)' }))
" --query "value[].message" -o tsv | tee output/shortpath-transport.txt


## 8. Post-Connection-Verifikation (optional)

Erst **nach** einer aktiven Verbindung aussagekräftig: Event ID **135** im Log `Microsoft-Windows-RemoteDesktopServices-RdpCoreCDV/Operational` meldet `transport type set to UDP`, wenn Shortpath genutzt wurde.

> Ohne vorherige Sitzung ist das Log leer – das ist kein Fehler. Die endgültige Bestätigung erfolgt sonst client-seitig im *Connection Information*-Dialog (`UDP` / `UDP (Relay)` / `UDP (Private Network)`).


In [ ]:
echo "=== Event ID 135 (RdpCoreCDV) auf $VM ==="
az vm run-command invoke -g $RG -n $VM --command-id RunPowerShellScript --scripts "
  try {
    Get-WinEvent -FilterHashtable @{LogName='Microsoft-Windows-RemoteDesktopServices-RdpCoreCDV/Operational';Id=135;StartTime=[DateTime]::Parse('$TEST_START_UTC').ToLocalTime()} -MaxEvents 200 -ErrorAction Stop |
      Where-Object Id -eq 135 |
      Select-Object TimeCreated, Message -First 5 | Format-List
  } catch { if (\$_.FullyQualifiedErrorId -like 'NoMatchingEventsFound*') { Write-Output 'Keine Transport-Events seit Testbeginn. Jetzt mit nativer Windows App verbinden.' } else { throw } }
" --query "value[].message" -o tsv | tee output/shortpath-event135.txt


## 9. Vergleich: Standard-Host vs. dedizierter Shortpath-Host

Hier werden **beide** Session-Hosts nebeneinander geprüft, um zu zeigen, ob RDP Shortpath ein- oder ausgeschaltet ist:

| Host | Pool | Zweck |
|------|------|-------|
| `vm-avd-` (vmavd01) | `hp-` | Produktiver Standard-Host |
| `vm-shortpath-` (vmsp01) | `hp-shortpath-` | Dedizierter Shortpath-Testhost |

Pro Host werden drei Dimensionen ausgewertet:

1. **Voraussetzung (UDP-Transport):** Session-Host-Registry `SelectTransport`. Nicht gesetzt = Windows-Default; `1` = nur TCP.
2. **Managed-Networks-Listener:** Registry `fUseUdpPortRedirector` (UDP 3390). In diesem Lab bewusst **aus** auf beiden Hosts.
3. **Tatsaechlich genutzt:** Nur Event **135** mit `transport type set to UDP` seit `TEST_START_UTC` belegt UDP-Nutzung. Ein Event 135 mit TCP ist kein Shortpath-Nachweis. Client-Dialog zur Zuordnung der eigenen Sitzung und fuer Multipath verwenden.

> Beide Hosts liegen im selben Spoke `10.1.0.0/16`. Die Service-Tag-UDR leitet `51.5.0.0/16` ueber NAT Gateway; die Firewall-Relay-Regel greift nur bei einem Pfad durch die Firewall. Direkte UDP-Pfade zu anderen Client-Adressen sind separat zu pruefen.
>
> Die endgültige Client-seitige Bestätigung erfolgt im AVD-Client unter *Connection Information* (`UDP` / `UDP (Relay)` / `UDP (Private Network)`).


In [ ]:
echo "=== Shortpath-Status im Vergleich (dauert ~1 Min für beide Hosts) ==="
echo ""
: > output/shortpath-compare.txt

for PAIR in "Standard-Host|$VM_BASE" "Shortpath-Host|$VM"; do
  LABEL="${PAIR%%|*}"; H="${PAIR##*|}"
  echo "######## $LABEL : $H ########" | tee -a output/shortpath-compare.txt
  az vm run-command invoke -g "$RG" -n "$H" --command-id RunPowerShellScript --scripts "
    \$p='HKLM:\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\Terminal Services'
    \$ErrorActionPreference='Stop'
    \$v=if (Test-Path \$p) { Get-ItemProperty -Path \$p } else { \$null }
    if (\$null -eq \$v -or \$null -eq \$v.SelectTransport) {
      Write-Output '1) Voraussetzung UDP-Transport : AKTIV (SelectTransport Default)'
    } elseif (\$v.SelectTransport -eq 1) {
      Write-Output '1) Voraussetzung UDP-Transport : AUS (SelectTransport=1 -> nur TCP)'
    } else {
      Write-Output ('1) UDP-Transport SelectTransport=' + \$v.SelectTransport)
    }
    Write-Output ('2) Managed-Networks-Listener   : ' + \$(if (\$v.fUseUdpPortRedirector) { 'AN (fUseUdpPortRedirector=' + \$v.fUseUdpPortRedirector + ')' } else { 'aus' }))
    try {
      \$events = @(Get-WinEvent -FilterHashtable @{LogName='Microsoft-Windows-RemoteDesktopServices-RdpCoreCDV/Operational';Id=135;StartTime=[DateTime]::Parse('$TEST_START_UTC').ToLocalTime()} -MaxEvents 200 -ErrorAction Stop)
      \$e = \$events | Where-Object Message -match 'transport type set to UDP\b' | Select-Object -First 1
      if (\$e) { Write-Output ('3) UDP seit Testbeginn belegt: ' + \$e.TimeCreated + '. Aktuelle Sitzung im Client pruefen.') }
      else { Write-Output '3) Kein UDP-Nachweis seit Testbeginn (nur andere Transport-Events).'; \$events | Select-Object -First 3 TimeCreated,Message | Format-List }
    } catch { if (\$_.FullyQualifiedErrorId -like 'NoMatchingEventsFound*') { Write-Output '3) Noch keine Transport-Events seit Testbeginn.' } else { throw } }
  " --query "value[].message" -o tsv | tee -a output/shortpath-compare.txt
  echo "" | tee -a output/shortpath-compare.txt
done


## 10. RDP Shortpath aktivieren / deaktivieren (In-Guest)

Public-Networks-Shortpath haengt am UDP-Transport. Auf dem Session Host steuert die Policy-Registry `SelectTransport` den Transport. Die Zelle ist optional und veraendert den ausgewaehlten Host.

| Wert | Bedeutung |
|------|-----------|
| nicht gesetzt / `2` | UDP und TCP erlaubt, Shortpath moeglich |
| `1` | UDP aus → RDP fällt auf **TCP** zurück, Shortpath deaktiviert |

`SelectTransport` unter `HKLM\SOFTWARE\Policies\Microsoft\Windows NT\Terminal Services` entspricht der Session-Host-Policy **Select RDP transport protocols**. `fClientDisableUDP` ist dagegen eine Client-Policy und kein Session-Host-Schalter.

**Steuerung:** unten `ACTION` und `TARGET` setzen, dann Zelle ausführen.

- `ACTION=disable` setzt `SelectTransport=1`
- `ACTION=enable` setzt `SelectTransport=2` (UDP und TCP)
- `ACTION=none` fuehrt keine Aenderung aus (Default)
- `TARGET=$VM` (Shortpath-Host) oder `TARGET=$VM_BASE` (Standard-Host)

> Nach Policy-Aenderungen den Testhost gemaess Microsoft-Dokumentation neu starten und eine neue Sitzung verbinden. Die Zelle startet den Host nicht automatisch neu. Verwaltete GPO-/Intune-Policies koennen lokale Aenderungen ueberschreiben.

### Anleitung zum Deaktivieren (offizielle Quellen)

Microsoft dokumentiert drei Wege, RDP Shortpath abzuschalten:

1. **Host-Pool-Ebene (empfohlen, kein In-Guest)** – per `Update-AzWvdHostPool` die UDP-Optionen auf `Disabled` setzen (`PublicUdp`/`RelayUdp` für Public Networks). Saubere, zentral steuerbare Variante: [Configure host pool networking settings](https://learn.microsoft.com/azure/virtual-desktop/configure-rdp-shortpath#configure-host-pool-networking-settings)
2. **Session-Host per GPO/Intune** – RDP-Shortpath-Typen auf **Disabled** setzen: [Configure RDP Shortpath via Intune/Group Policy](https://learn.microsoft.com/azure/virtual-desktop/configure-rdp-shortpath#configure-rdp-shortpath-for-public-networks-using-microsoft-intune-and-group-policy)
3. **UDP-Transport komplett aus** – „Select RDP transport protocols" → **Use only TCP**. Auf dem **Session Host** ist das der Registry-Wert `SelectTransport=1` (siehe [Check that UDP is enabled on session hosts](https://learn.microsoft.com/azure/virtual-desktop/configure-rdp-shortpath#check-that-udp-is-enabled-on-session-hosts)); auf dem **Client** der Wert `fClientDisableUDP=1` unter `...\Terminal Services\Client` (siehe [Check that UDP is enabled on Windows client devices](https://learn.microsoft.com/azure/virtual-desktop/configure-rdp-shortpath#check-that-udp-is-enabled-on-windows-client-devices)).

> Fuer einen zentral gesteuerten Vergleich koennen alternativ die Host-Pool-UDP-Optionen verwendet werden. TURN nach einem Isolationstest wieder aktivieren, damit der Relay-Fallback erhalten bleibt.

**Allgemeine Referenzen:**

- [Configure RDP Shortpath – inkl. „verify it is working" und „disable it if needed"](https://learn.microsoft.com/azure/virtual-desktop/configure-rdp-shortpath)
- [RDP Shortpath – Funktionsweise (STUN/TURN, public vs. managed networks)](https://learn.microsoft.com/azure/virtual-desktop/rdp-shortpath)
- [Policy CSP – ADMX_TerminalServer/TS_SELECT_TRANSPORT (UDP komplett abschalten → „Use only TCP")](https://learn.microsoft.com/windows/client-management/mdm/policy-csp-admx-terminalserver#ts_select_transport)
- [Use the administrative template for Azure Virtual Desktop (GPO/Intune Setup)](https://learn.microsoft.com/azure/virtual-desktop/administrative-template)


In [ ]:
# ----- Steuerung -----
ACTION=none           # none | disable | enable (bewusste Auswahl erforderlich)
TARGET=$VM            # $VM = Shortpath-Host (vmsp01) | $VM_BASE = Standard-Host (vmavd01)
# ---------------------

echo "=== Shortpath '$ACTION' auf $TARGET (dauert ~30s) ==="
if [ "$ACTION" = "disable" ]; then
  PS="
    \$p='HKLM:\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\Terminal Services'
    New-Item -Path \$p -Force | Out-Null
    New-ItemProperty -Path \$p -Name SelectTransport -PropertyType DWord -Value 1 -Force | Out-Null
    Write-Output 'SelectTransport=1 (nur TCP). Testhost vor dem Vergleich neu starten.'
  "
elif [ "$ACTION" = "enable" ]; then
  PS="
    \$p='HKLM:\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\Terminal Services'
    New-Item -Path \$p -Force | Out-Null
    New-ItemProperty -Path \$p -Name SelectTransport -PropertyType DWord -Value 2 -Force | Out-Null
    Write-Output 'SelectTransport=2 (UDP und TCP). Testhost vor dem Vergleich neu starten.'
  "
elif [ "$ACTION" = "none" ]; then
  echo 'Keine Aenderung ausgewaehlt.'; PS=""
else
  echo "Unbekannte ACTION '$ACTION' (erlaubt: disable | enable)"; PS=""
fi

if [ -n "$PS" ]; then
  az vm run-command invoke -g "$RG" -n "$TARGET" --command-id RunPowerShellScript \
    --scripts "$PS" --query "value[].message" -o tsv | tee output/shortpath-toggle.txt
fi
